# 06 — Real-Data Learning Curve for Drone–Bird Classification

This notebook establishes the **real-only reference learning curve** for the
project *Synthetic Radar Data Augmentation for Limited-Data Micro-Doppler
Classification*.

Four nested, class-balanced subsets of the official real training partition are
evaluated: **10%, 25%, 50%, and 100%**. The CNN architecture, preprocessing,
random seed, validation partition, test partition, and evaluation protocol are
held fixed. Only the number of real training samples changes.

> **Safe rerun policy:** all completed experiments use `RUN_TRAINING = False`.
> A top-to-bottom rerun therefore loads saved checkpoints and training histories
> instead of fitting the models again. Set the flag to `True` only for a genuinely
> missing experiment after confirming that no checkpoint or test result exists.

## 1. Experimental Protocol

- **Task:** binary classification of bird (`0`) versus drone (`1`).
- **Training data:** nested, balanced subsets of the official real training split.
- **Validation data:** the fixed official real validation split, used for early
  stopping and threshold selection.
- **Test data:** the fixed official real test split, used once per trained model
  for final evaluation.
- **Primary threshold objective:** maximum validation macro-F1, with balanced
  accuracy as the tie-breaker.
- **Primary reporting metrics:** macro-F1, balanced accuracy, bird recall, drone
  recall, and ROC-AUC.

The official validation and test sets preserve their natural class imbalance.
No validation or test sample is used for model fitting. The results are
segment-level estimates under the official split; session-independent evaluation
is a separate robustness experiment.

## 2. Imports and Reproducibility

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from tensorflow.keras import layers, models, regularizers

In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as error:
    print("Deterministic operations unavailable:", error)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 3. Project Paths and Required Files

The notebook expects to be executed from the project's `notebooks` directory. Existing experiment artifacts are checked before training to prevent accidental overwriting.

In [ ]:
OFFICIAL_DATA_DIR = Path("../data/processed/official_split")
LIMITED_DATA_DIR = Path("../data/processed/limited_subsets")
OUTPUT_DIR = Path("../outputs/baseline_classification")
CHECKPOINT_DIR = Path("../checkpoints/baseline")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    OFFICIAL_DATA_DIR / "X_train.npy",
    OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "X_validation.npy",
    OFFICIAL_DATA_DIR / "y_validation.npy",
    OFFICIAL_DATA_DIR / "X_test.npy",
    OFFICIAL_DATA_DIR / "y_test.npy",
    OFFICIAL_DATA_DIR / "metadata_test.csv",
    *[LIMITED_DATA_DIR / f"indices_{name}.npy" for name in
      ["10_percent", "25_percent", "50_percent", "100_percent"]]
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(map(str, missing_files)))

print("All required files were found.")
print("Processed data:", OFFICIAL_DATA_DIR.resolve())
print("Limited subsets:", LIMITED_DATA_DIR.resolve())

## 4. Saved-Experiment Integrity Check

In [ ]:
SUBSETS = ["10_percent", "25_percent", "50_percent", "100_percent"]
status_records = []

for subset_name in SUBSETS:
    result_dir = OUTPUT_DIR / f"{subset_name}_seed_{RANDOM_SEED}"
    checkpoint = CHECKPOINT_DIR / f"baseline_{subset_name}_seed_{RANDOM_SEED}.keras"
    metrics = result_dir / "test_metrics.csv"
    history = result_dir / "training_history.csv"
    status_records.append({
        "subset": subset_name,
        "result_directory": result_dir.exists(),
        "test_metrics": metrics.exists(),
        "training_history": history.exists(),
        "checkpoint": checkpoint.exists(),
        "experiment_complete": metrics.exists() and history.exists() and checkpoint.exists()
    })

experiment_status_df = pd.DataFrame(status_records)
display(experiment_status_df)

All four real-only experiments should be complete before the final learning
curve is assembled. A completed experiment requires its checkpoint, training
history, and test-metrics file. The status table above is the first safeguard
against accidental retraining or incomplete comparisons.

The earlier no-bias 25% pilot is not part of this notebook or the learning curve;
all reported models use the verified 29,121-parameter architecture.

## 5. Load Fixed Data Partitions

In [ ]:
X_train_complete = np.load(OFFICIAL_DATA_DIR / "X_train.npy", mmap_mode="r")
y_train_complete = np.load(OFFICIAL_DATA_DIR / "y_train.npy", mmap_mode="r")
X_validation_fixed = np.load(OFFICIAL_DATA_DIR / "X_validation.npy", mmap_mode="r")
y_validation_fixed = np.load(OFFICIAL_DATA_DIR / "y_validation.npy")
X_test_fixed = np.load(OFFICIAL_DATA_DIR / "X_test.npy", mmap_mode="r")
y_test_fixed = np.load(OFFICIAL_DATA_DIR / "y_test.npy")
metadata_test = pd.read_csv(OFFICIAL_DATA_DIR / "metadata_test.csv")

print("Base training tensors:", X_train_complete.shape)
print("Validation tensors:", X_validation_fixed.shape)
print("Test tensors:", X_test_fixed.shape)
print("Test metadata:", metadata_test.shape)

assert len(X_validation_fixed) == len(y_validation_fixed)
assert len(X_test_fixed) == len(y_test_fixed) == len(metadata_test)

In [ ]:
BATCH_SIZE = 64

validation_dataset_fixed = (tf.data.Dataset
    .from_tensor_slices((X_validation_fixed[..., np.newaxis], y_validation_fixed))
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

test_dataset_fixed = (tf.data.Dataset
    .from_tensor_slices((X_test_fixed[..., np.newaxis], y_test_fixed))
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print("Validation batches:", len(validation_dataset_fixed))
print("Test batches:", len(test_dataset_fixed))

## 6. Fixed CNN Architecture

All convolutional layers retain their bias parameters, matching the original
10% baseline. The architecture must contain exactly **29,121 parameters**
(28,897 trainable and 224 non-trainable). This assertion prevents an accidental
architecture change from being interpreted as a learning-curve effect.

In [ ]:
def build_baseline_model(input_shape=(5, 150, 1)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, (3, 7), padding="same"),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.Conv2D(32, (3, 5), padding="same"),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )
    return model

In [ ]:
architecture_check = build_baseline_model()
assert architecture_check.count_params() == 29121
print("Architecture verified:", architecture_check.count_params(), "parameters")
del architecture_check
tf.keras.backend.clear_session()

## 7. 25% Real-Data Baseline

In [ ]:
SELECTED_SUBSET = "25_percent"
RESULT_DIR = OUTPUT_DIR / f"{SELECTED_SUBSET}_seed_{RANDOM_SEED}"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"baseline_{SELECTED_SUBSET}_seed_{RANDOM_SEED}.keras"

subset_indices = np.load(LIMITED_DATA_DIR / f"indices_{SELECTED_SUBSET}.npy")
X_train_selected = np.asarray(X_train_complete[subset_indices], dtype=np.float32)[..., np.newaxis]
y_train_selected = np.asarray(y_train_complete[subset_indices], dtype=np.uint8)

print("Training tensor:", X_train_selected.shape)
print("Class counts [bird, drone]:", np.bincount(y_train_selected))
assert X_train_selected.shape == (2876, 5, 150, 1)
assert np.array_equal(np.bincount(y_train_selected), [1438, 1438])
assert np.isfinite(X_train_selected).all()
assert 0.0 <= X_train_selected.min() <= X_train_selected.max() <= 1.0
print("The 25% subset passed all validation checks.")

In [ ]:
training_dataset_selected = (tf.data.Dataset
    .from_tensor_slices((X_train_selected, y_train_selected))
    .shuffle(len(y_train_selected), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
print("Training batches:", len(training_dataset_selected))

### 7.1 Protected Training or Checkpoint Recovery

The corrected 25% model is already trained, so `RUN_TRAINING` defaults to
`False`. In this mode the saved checkpoint and history are loaded. Training is
allowed only when both the checkpoint and final test metrics are absent.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    if CHECKPOINT_PATH.exists() or (RESULT_DIR / "test_metrics.csv").exists():
        raise FileExistsError("Existing experiment detected; training stopped to prevent overwrite.")
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)
    model_selected = build_baseline_model()
    callbacks_selected = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            CHECKPOINT_PATH, monitor="val_loss", save_best_only=True, verbose=1),
        tf.keras.callbacks.CSVLogger(RESULT_DIR / "training_log.csv")
    ]
    history_selected = model_selected.fit(
        training_dataset_selected, validation_data=validation_dataset_fixed,
        epochs=50, callbacks=callbacks_selected, verbose=1)
    history_selected_df = pd.DataFrame(history_selected.history)
    history_selected_df.insert(0, "epoch", np.arange(1, len(history_selected_df) + 1))
    history_selected_df.to_csv(RESULT_DIR / "training_history.csv", index=False)
else:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError("The saved 25% checkpoint was not found.")
    model_selected = tf.keras.models.load_model(CHECKPOINT_PATH)
    history_selected_df = pd.read_csv(RESULT_DIR / "training_history.csv")
    if "epoch" not in history_selected_df.columns:
        history_selected_df.insert(0, "epoch", np.arange(1, len(history_selected_df) + 1))
    print("Saved 25% checkpoint loaded; no training was performed.")

assert model_selected.count_params() == 29121

### 7.2 Training Behaviour

In [ ]:
best_epoch = int(history_selected_df.loc[history_selected_df["val_loss"].idxmin(), "epoch"])
best_validation_loss = float(history_selected_df["val_loss"].min())
print("Completed epochs:", len(history_selected_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_validation_loss)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for axis, metric, title in zip(
    axes, ["loss", "accuracy", "roc_auc"],
    ["Binary Cross-Entropy Loss", "Accuracy", "ROC-AUC"]):
    axis.plot(history_selected_df["epoch"], history_selected_df[metric], label="Training")
    axis.plot(history_selected_df["epoch"], history_selected_df[f"val_{metric}"], label="Validation")
    axis.axvline(best_epoch, color="black", linestyle="--", label=f"Best epoch: {best_epoch}")
    axis.set_title(title); axis.set_xlabel("Epoch"); axis.grid(alpha=0.2); axis.legend()
plt.show()

### 7.3 Validation-Only Threshold Selection

In [ ]:
validation_probabilities = model_selected.predict(validation_dataset_fixed, verbose=1).reshape(-1)
threshold_records = []

for threshold in np.linspace(0.01, 0.99, 199):
    predictions = (validation_probabilities >= threshold).astype(np.uint8)
    threshold_records.append({
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_validation_fixed, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation_fixed, predictions),
        "macro_f1": f1_score(y_validation_fixed, predictions, average="macro", zero_division=0),
        "bird_recall": np.mean(predictions[y_validation_fixed == 0] == 0),
        "drone_recall": np.mean(predictions[y_validation_fixed == 1] == 1)
    })

threshold_search_df = pd.DataFrame(threshold_records)
best_threshold_row = threshold_search_df.sort_values(
    ["macro_f1", "balanced_accuracy"], ascending=False).iloc[0]
optimal_threshold = float(best_threshold_row["threshold"])
print("Probability range:", validation_probabilities.min(), "to", validation_probabilities.max())
print("Selected threshold:", optimal_threshold)
display(best_threshold_row.to_frame().T.round(4))

### Validation Threshold Result

| Strategy | Threshold | Accuracy | Balanced accuracy | Macro-F1 | Bird recall | Drone recall |
|---|---:|---:|---:|---:|---:|---:|
| Default | 0.5000 | 0.9768 | 0.9536 | 0.9525 | 0.9212 | 0.9860 |
| Validation-optimized | 0.4208 | 0.9780 | 0.9492 | 0.9542 | 0.9091 | 0.9893 |

The optimized threshold gives the highest macro-F1, although threshold 0.5 has slightly higher balanced accuracy and bird recall. This trade-off is retained because the selection rule was specified before test evaluation.

### 7.4 Untouched Test-Set Evaluation

In [ ]:
saved_metrics_path = RESULT_DIR / "test_metrics.csv"
if saved_metrics_path.exists():
    optimal_threshold = float(pd.read_csv(saved_metrics_path).loc[0, "threshold"])

test_probabilities = model_selected.predict(test_dataset_fixed, verbose=1).reshape(-1)
test_predictions = (test_probabilities >= optimal_threshold).astype(np.uint8)
test_report = classification_report(
    y_test_fixed, test_predictions, target_names=["bird", "drone"],
    output_dict=True, zero_division=0)

test_metrics_25_df = pd.DataFrame([{
    "subset": SELECTED_SUBSET, "seed": RANDOM_SEED,
    "training_samples": len(y_train_selected), "samples_per_class": int(np.sum(y_train_selected == 0)),
    "best_epoch": best_epoch, "best_validation_loss": best_validation_loss,
    "threshold": optimal_threshold,
    "accuracy": accuracy_score(y_test_fixed, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test_fixed, test_predictions),
    "macro_f1": f1_score(y_test_fixed, test_predictions, average="macro", zero_division=0),
    "bird_precision": test_report["bird"]["precision"],
    "bird_recall": test_report["bird"]["recall"],
    "bird_f1": test_report["bird"]["f1-score"],
    "drone_precision": test_report["drone"]["precision"],
    "drone_recall": test_report["drone"]["recall"],
    "drone_f1": test_report["drone"]["f1-score"],
    "roc_auc": roc_auc_score(y_test_fixed, test_probabilities)
}])
display(test_metrics_25_df.round(4))
print(classification_report(y_test_fixed, test_predictions,
                            target_names=["bird", "drone"], digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test_fixed, test_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Bird", "Drone"], yticklabels=["Bird", "Drone"])
plt.title("Test Confusion Matrix — 25% Real-Data Baseline")
plt.xlabel("Predicted target"); plt.ylabel("True target")
plt.tight_layout(); plt.show()

### 25% Test Conclusion

The 25% model achieved **97.97% accuracy**, **95.22% balanced accuracy**, **95.79% macro-F1**, and **99.62% ROC-AUC**. Bird precision, recall, and F1 were 94.21%, 91.37%, and 92.77%; drone precision, recall, and F1 were 98.57%, 99.07%, and 98.82%. The confusion matrix contains 911 correctly classified birds, 86 birds classified as drones, 56 drones classified as birds, and 5,943 correctly classified drones.

### 7.5 Performance by Original Target Subtype

In [ ]:
test_results_25_df = metadata_test.copy()
test_results_25_df["true_binary_label"] = y_test_fixed
test_results_25_df["drone_probability"] = test_probabilities
test_results_25_df["predicted_binary_label"] = test_predictions
test_results_25_df["true_target_group"] = np.where(y_test_fixed == 1, "drone", "bird")
test_results_25_df["correct"] = y_test_fixed == test_predictions

subtype_metrics_25 = (test_results_25_df
    .groupby(["true_target_group", "original_label"], observed=True)
    .agg(samples=("correct", "size"), correct_predictions=("correct", "sum"),
         recall=("correct", "mean"),
         mean_drone_probability=("drone_probability", "mean"),
         median_drone_probability=("drone_probability", "median"))
    .reset_index().sort_values(["true_target_group", "recall"]))
display(subtype_metrics_25.style.format({
    "recall": "{:.2%}", "mean_drone_probability": "{:.4f}",
    "median_drone_probability": "{:.4f}"}))

### Subtype Findings

All drone models reached at least 97.10% recall. D1 improved from 70.20% in the 10% experiment to 97.70%, while D2 and D4 reached 100%. Seagull recall increased to 86.14% and black-headed gull recall to 94.82%. Heron recall was 82.22%, but its support is only 45. Pigeon and raven contain only four and one test samples, so their apparent 100% recalls are not reliable species-level evidence.

### 7.6 Performance by Target Range

In [ ]:
RANGE_BINS = [0, 30, 50, 75, np.inf]
RANGE_LABELS = ["<30 m", "30–50 m", "50–75 m", ">75 m"]
test_results_25_df["range_bin"] = pd.cut(
    test_results_25_df["range_m"], bins=RANGE_BINS,
    labels=RANGE_LABELS, right=False, include_lowest=True)

range_records = []
for range_label in RANGE_LABELS:
    group = test_results_25_df[test_results_25_df["range_bin"] == range_label]
    yt = group["true_binary_label"].to_numpy()
    yp = group["predicted_binary_label"].to_numpy()
    prob = group["drone_probability"].to_numpy()
    range_records.append({
        "range_bin": range_label, "samples": len(group),
        "bird_samples": int(np.sum(yt == 0)), "drone_samples": int(np.sum(yt == 1)),
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "bird_recall": np.mean(yp[yt == 0] == 0),
        "drone_recall": np.mean(yp[yt == 1] == 1),
        "roc_auc": roc_auc_score(yt, prob),
        "mean_drone_probability": np.mean(prob)
    })
range_metrics_25 = pd.DataFrame(range_records)
display(range_metrics_25.round(4))

In [ ]:
range_plot = range_metrics_25.melt(
    id_vars="range_bin",
    value_vars=["bird_recall", "drone_recall", "balanced_accuracy"],
    var_name="metric", value_name="score")
plt.figure(figsize=(11, 6))
sns.barplot(data=range_plot, x="range_bin", y="score", hue="metric", order=RANGE_LABELS)
plt.axhline(0.5, color="black", linestyle="--", linewidth=1, label="50% reference")
plt.ylim(0, 1.05); plt.xlabel("Target range interval"); plt.ylabel("Score")
plt.title("Test Performance by Target Range — 25% Real-Data Baseline")
plt.grid(axis="y", alpha=0.25); plt.tight_layout(); plt.show()

### Range-Robustness Conclusion

Balanced accuracy exceeded 93% in all four range intervals. Compared with the 10% model, bird recall below 30 m increased from 49.52% to 93.33%, while drone recall beyond 75 m increased from 51.79% to 96.73%. The severe range-dependent failures were therefore largely corrected by increasing the real training set. This suggests that those failures were mainly caused by insufficient coverage rather than an unavoidable limitation of the range–Doppler representation.

### 7.7 Direct 10%–25% Comparison

In [ ]:
metrics_10 = pd.read_csv(OUTPUT_DIR / "10_percent_seed_42" / "test_metrics.csv")
metrics_25 = test_metrics_25_df.copy()
comparison = pd.concat([metrics_10, metrics_25], ignore_index=True)
display(comparison[[
    "subset", "threshold", "accuracy", "balanced_accuracy", "macro_f1",
    "bird_precision", "bird_recall", "bird_f1",
    "drone_recall", "roc_auc"
]].round(4))

| Metric | 10% baseline | 25% baseline | Absolute change |
|---|---:|---:|---:|
| Accuracy | 0.8942 | 0.9797 | +0.0855 |
| Balanced accuracy | 0.8497 | 0.9522 | +0.1025 |
| Macro-F1 | 0.8082 | 0.9579 | +0.1497 |
| ROC-AUC | 0.9353 | 0.9962 | +0.0609 |
| Bird precision | 0.5979 | 0.9421 | +0.3442 |
| Bird recall | 0.7874 | 0.9137 | +0.1263 |
| Bird F1 | 0.6797 | 0.9277 | +0.2480 |

Increasing the balanced training set from 575 to 1,438 samples per class produced a large improvement, especially for bird classification and range robustness. The 25% real-only result becomes an important target for synthetic augmentation: augmentation of the 10% subset should ideally approach it without requiring the additional real observations.

### 7.8 Saved Reproducibility Artifacts

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
test_metrics_25_df.to_csv(RESULT_DIR / "test_metrics.csv", index=False)
test_results_25_df.to_csv(RESULT_DIR / "test_predictions.csv", index=False)
subtype_metrics_25.to_csv(RESULT_DIR / "subtype_metrics.csv", index=False)
range_metrics_25.to_csv(RESULT_DIR / "range_metrics.csv", index=False)
threshold_search_df.to_csv(RESULT_DIR / "validation_threshold_search.csv", index=False)

configuration = {
    "subset": SELECTED_SUBSET,
    "random_seed": RANDOM_SEED,
    "architecture_parameters": 29121,
    "training_samples": int(len(y_train_selected)),
    "samples_per_class": int(np.sum(y_train_selected == 0)),
    "completed_epochs": int(len(history_selected_df)),
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_validation_loss),
    "decision_threshold": float(optimal_threshold),
    "threshold_selection": "Maximum validation macro-F1 with balanced-accuracy tie-break"
}
with open(RESULT_DIR / "experiment_config.json", "w", encoding="utf-8") as file:
    json.dump(configuration, file, indent=4)
print("25% experiment artifacts saved to:", RESULT_DIR.resolve())

### 7.9 Interim Conclusion

The 25% CNN substantially outperformed the 10% limited-data baseline and
corrected most subtype- and range-specific failures. This is the largest step in
the real-data learning curve and provides the main performance target for future
synthetic augmentation of the 10% subset.

## 8. 50% Real-Data Baseline

The 50% experiment uses 2,875 bird and 2,875 drone segments. Every other
experimental setting remains fixed.

### 8.1 Configuration and Rerun Protection

In [ ]:
SELECTED_SUBSET = "50_percent"
RUN_TRAINING = False

RESULT_DIR = (
    OUTPUT_DIR
    / f"{SELECTED_SUBSET}_seed_{RANDOM_SEED}"
)

CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / f"baseline_{SELECTED_SUBSET}_seed_{RANDOM_SEED}.keras"
)

print("Selected subset:", SELECTED_SUBSET)
print("Result directory:", RESULT_DIR.resolve())
print("Checkpoint:", CHECKPOINT_PATH.resolve())
print("Existing checkpoint:", CHECKPOINT_PATH.exists())

### 8.2 Overwrite Guard

In [ ]:
if RUN_TRAINING:
    if CHECKPOINT_PATH.exists():
        raise FileExistsError(
            "The 50% checkpoint already exists."
        )

    if (RESULT_DIR / "test_metrics.csv").exists():
        raise FileExistsError(
            "The 50% test results already exist."
        )

    RESULT_DIR.mkdir(
        parents=True,
        exist_ok=False
    )

print("The 50% experiment is ready.")

### 8.3 Load and Validate the Balanced Subset

In [ ]:
subset_indices = np.load(
    LIMITED_DATA_DIR
    / f"indices_{SELECTED_SUBSET}.npy"
)

X_train_selected = np.asarray(
    X_train_complete[subset_indices],
    dtype=np.float32
)[..., np.newaxis]

y_train_selected = np.asarray(
    y_train_complete[subset_indices],
    dtype=np.uint8
)

print(
    "Training tensor:",
    X_train_selected.shape
)

print(
    "Training labels:",
    y_train_selected.shape
)

print(
    "Class counts [bird, drone]:",
    np.bincount(y_train_selected)
)

assert X_train_selected.shape == (
    5750,
    5,
    150,
    1
)

assert np.array_equal(
    np.bincount(y_train_selected),
    [2875, 2875]
)

assert np.isfinite(
    X_train_selected
).all()

assert X_train_selected.min() >= 0.0
assert X_train_selected.max() <= 1.0

print(
    "The 50% subset passed all checks."
)

### 8.4 Build the Training Dataset

In [ ]:
training_dataset_selected = (
    tf.data.Dataset
    .from_tensor_slices((
        X_train_selected,
        y_train_selected
    ))
    .shuffle(
        buffer_size=len(
            y_train_selected
        ),
        seed=RANDOM_SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(
    "Training batches:",
    len(training_dataset_selected)
)

### 8.5 Verify the Model Architecture

In [ ]:
tf.keras.backend.clear_session()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

model_selected = build_baseline_model()

assert model_selected.count_params() == 29121

print(
    "Architecture verified:",
    model_selected.count_params(),
    "parameters"
)

### 8.6 Training Callbacks

In [ ]:
callbacks_selected = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        RESULT_DIR / "training_log.csv"
    )
]

### 8.7 Train or Load the Saved Model

In [ ]:
if RUN_TRAINING:
    history_selected = model_selected.fit(
        training_dataset_selected,
        validation_data=validation_dataset_fixed,
        epochs=50,
        callbacks=callbacks_selected,
        verbose=1
    )
    history_selected_df = pd.DataFrame(history_selected.history)
    history_selected_df.insert(
        0, "epoch", np.arange(1, len(history_selected_df) + 1)
    )
    history_selected_df.to_csv(
        RESULT_DIR / "training_history.csv", index=False
    )
else:
    history_path = RESULT_DIR / "training_history.csv"
    if not CHECKPOINT_PATH.exists() or not history_path.exists():
        raise FileNotFoundError(
            "The saved 50% checkpoint or training history was not found."
        )
    model_selected = tf.keras.models.load_model(CHECKPOINT_PATH)
    history_selected_df = pd.read_csv(history_path)
    if "epoch" not in history_selected_df.columns:
        history_selected_df.insert(
            0, "epoch", np.arange(1, len(history_selected_df) + 1)
        )
    print("Saved 50% checkpoint loaded; no training was performed.")

assert model_selected.count_params() == 29121

### 8.8 Training Summary

In [ ]:
best_epoch = int(
    history_selected_df.loc[history_selected_df["val_loss"].idxmin(), "epoch"]
)
best_validation_loss = float(history_selected_df["val_loss"].min())

print("Completed epochs:", len(history_selected_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_validation_loss)
print("Checkpoint available:", CHECKPOINT_PATH.exists())

### 8.9 50% Training Results

The 50% baseline was trained using 5,750 balanced real radar segments:

- 2,875 bird segments
- 2,875 drone segments
- 90 batches per epoch
- Maximum of 50 epochs
- Early-stopping patience of 8 epochs

Training stopped after epoch 37 and restored the model weights from epoch 29, which achieved the minimum validation loss.

### Best-Epoch Results

| Metric | Result |
|---|---:|
| Best epoch | 29 |
| Training loss | 0.0140 |
| Training accuracy | 0.9963 |
| Training ROC-AUC | 1.0000 |
| Validation loss | 0.0541 |
| Validation accuracy | 0.9843 |
| Validation ROC-AUC | 0.9965 |
| Validation PR-AUC | 0.9993 |

The best validation loss improved from approximately **0.0619 for the 25% model** to **0.0541 for the 50% model**.

Training performance approached 100%, while validation performance fluctuated substantially across epochs. This suggests that the model can fit the balanced training subset almost perfectly, but its fixed-threshold validation predictions remain sensitive to changes in the learned probability distribution.

Early stopping was therefore important because the final training epoch was not the best validation epoch. The restored epoch-29 model is used in all subsequent analyses.

In [ ]:
validation_probabilities_50 = (
    model_selected
    .predict(
        validation_dataset_fixed,
        verbose=1
    )
    .reshape(-1)
)

print(
    "Minimum validation probability:",
    float(
        validation_probabilities_50.min()
    )
)

print(
    "Maximum validation probability:",
    float(
        validation_probabilities_50.max()
    )
)

print(
    "Mean validation probability:",
    float(
        validation_probabilities_50.mean()
    )
)

print(
    "Predictions below 0.5:",
    int(
        np.sum(
            validation_probabilities_50 < 0.5
        )
    )
)

print(
    "Predictions at or above 0.5:",
    int(
        np.sum(
            validation_probabilities_50 >= 0.5
        )
    )
)

### 8.10 Search Validation Thresholds

In [ ]:
threshold_records_50 = []

for threshold in np.linspace(
    0.01,
    0.99,
    199
):
    predictions = (
        validation_probabilities_50
        >= threshold
    ).astype(np.uint8)

    threshold_records_50.append({
        "threshold": float(threshold),

        "accuracy": accuracy_score(
            y_validation_fixed,
            predictions
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_fixed,
                predictions
            ),

        "macro_f1": f1_score(
            y_validation_fixed,
            predictions,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            predictions[
                y_validation_fixed == 0
            ] == 0
        ),

        "drone_recall": np.mean(
            predictions[
                y_validation_fixed == 1
            ] == 1
        )
    })

threshold_search_50_df = pd.DataFrame(
    threshold_records_50
)

best_threshold_50_row = (
    threshold_search_50_df
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy"
        ],
        ascending=False
    )
    .iloc[0]
)

optimal_threshold_50 = float(
    best_threshold_50_row["threshold"]
)

print(
    "Selected validation threshold:",
    optimal_threshold_50
)

display(
    best_threshold_50_row
    .to_frame()
    .T
    .round(4)
)

### 8.11 Compare the Selected Threshold with 0.5

In [ ]:
threshold_comparison_50_records = []

for threshold_name, threshold in [
    ("default_0.5", 0.5),
    (
        "validation_optimised",
        optimal_threshold_50
    )
]:
    predictions = (
        validation_probabilities_50
        >= threshold
    ).astype(np.uint8)

    threshold_comparison_50_records.append({
        "threshold_name": threshold_name,
        "threshold": threshold,

        "accuracy": accuracy_score(
            y_validation_fixed,
            predictions
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_fixed,
                predictions
            ),

        "macro_f1": f1_score(
            y_validation_fixed,
            predictions,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            predictions[
                y_validation_fixed == 0
            ] == 0
        ),

        "drone_recall": np.mean(
            predictions[
                y_validation_fixed == 1
            ] == 1
        )
    })

threshold_comparison_50_df = pd.DataFrame(
    threshold_comparison_50_records
)

display(
    threshold_comparison_50_df.round(4)
)

In [ ]:
threshold_search_50_df.to_csv(
    RESULT_DIR
    / "validation_threshold_search.csv",
    index=False
)

threshold_comparison_50_df.to_csv(
    RESULT_DIR
    / "validation_threshold_comparison.csv",
    index=False
)

print("50% validation threshold results saved.")

### 8.12 Validation Threshold Analysis

The restored 50% model produces validation probabilities across almost the complete probability interval:

- Minimum probability: approximately 0.00000007
- Maximum probability: 1.0000
- Mean probability: 0.8537
- Predictions below 0.5: 1,016
- Predictions at or above 0.5: 5,972

### Threshold Comparison

| Strategy | Threshold | Accuracy | Balanced accuracy | Macro-F1 | Bird recall | Drone recall |
|---|---:|---:|---:|---:|---:|---:|
| Default | 0.5000 | 0.9843 | 0.9731 | 0.9680 | 0.9576 | 0.9887 |
| Validation-optimized | 0.4951 | 0.9844 | 0.9732 | 0.9683 | 0.9576 | 0.9888 |

The validation-optimized threshold of **0.4951** is almost identical to the standard threshold of 0.5. Its improvements are very small, corresponding to approximately one additional correctly classified validation sample.

This indicates that the 50% model has strong class separation and substantially better probability calibration than the 10% model.

According to the predefined macro-F1 selection rule, the threshold of **0.4951** is retained and locked before test evaluation.

### 8.13 Evaluate the Untouched Test Set

In [ ]:
LOCKED_THRESHOLD_50 = float(
    optimal_threshold_50
)

print(
    "Locked 50% threshold:",
    LOCKED_THRESHOLD_50
)

test_probabilities_50 = (
    model_selected
    .predict(
        test_dataset_fixed,
        verbose=1
    )
    .reshape(-1)
)

test_predictions_50 = (
    test_probabilities_50
    >= LOCKED_THRESHOLD_50
).astype(np.uint8)

print(
    "Minimum test probability:",
    float(test_probabilities_50.min())
)

print(
    "Maximum test probability:",
    float(test_probabilities_50.max())
)

print(
    "Mean test probability:",
    float(test_probabilities_50.mean())
)

### 8.14 Calculate Test Metrics

In [ ]:
test_report_50 = classification_report(
    y_test_fixed,
    test_predictions_50,
    target_names=[
        "bird",
        "drone"
    ],
    output_dict=True,
    zero_division=0
)

test_metrics_50_df = pd.DataFrame([{
    "subset": SELECTED_SUBSET,
    "seed": RANDOM_SEED,
    "training_samples": len(
        y_train_selected
    ),
    "samples_per_class": int(
        np.sum(y_train_selected == 0)
    ),
    "best_epoch": best_epoch,
    "best_validation_loss":
        best_validation_loss,
    "threshold": LOCKED_THRESHOLD_50,

    "accuracy": accuracy_score(
        y_test_fixed,
        test_predictions_50
    ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test_fixed,
            test_predictions_50
        ),

    "macro_f1": f1_score(
        y_test_fixed,
        test_predictions_50,
        average="macro",
        zero_division=0
    ),

    "bird_precision":
        test_report_50["bird"]["precision"],

    "bird_recall":
        test_report_50["bird"]["recall"],

    "bird_f1":
        test_report_50["bird"]["f1-score"],

    "drone_precision":
        test_report_50["drone"]["precision"],

    "drone_recall":
        test_report_50["drone"]["recall"],

    "drone_f1":
        test_report_50["drone"]["f1-score"],

    "roc_auc": roc_auc_score(
        y_test_fixed,
        test_probabilities_50
    )
}])

display(
    test_metrics_50_df.round(4)
)

print(
    classification_report(
        y_test_fixed,
        test_predictions_50,
        target_names=[
            "bird",
            "drone"
        ],
        digits=4,
        zero_division=0
    )
)

### 8.15 Plot the Confusion Matrix

In [ ]:
confusion_matrix_50 = confusion_matrix(
    y_test_fixed,
    test_predictions_50
)

display(
    pd.DataFrame(
        confusion_matrix_50,
        index=[
            "True bird",
            "True drone"
        ],
        columns=[
            "Predicted bird",
            "Predicted drone"
        ]
    )
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    confusion_matrix_50,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Bird",
        "Drone"
    ],
    yticklabels=[
        "Bird",
        "Drone"
    ]
)

plt.title(
    "Test Confusion Matrix — 50% Real-Data Baseline"
)

plt.xlabel("Predicted target")
plt.ylabel("True target")
plt.tight_layout()
plt.show()

In [ ]:
test_results_50_df = metadata_test.copy()

test_results_50_df[
    "true_binary_label"
] = y_test_fixed

test_results_50_df[
    "drone_probability"
] = test_probabilities_50

test_results_50_df[
    "predicted_binary_label"
] = test_predictions_50

test_results_50_df[
    "true_target_group"
] = np.where(
    y_test_fixed == 1,
    "drone",
    "bird"
)

test_results_50_df[
    "correct"
] = (
    y_test_fixed
    == test_predictions_50
)

test_metrics_50_df.to_csv(
    RESULT_DIR / "test_metrics.csv",
    index=False
)

test_results_50_df.to_csv(
    RESULT_DIR / "test_predictions.csv",
    index=False
)

experiment_config_50 = {
    "subset": SELECTED_SUBSET,
    "random_seed": RANDOM_SEED,
    "architecture_parameters": 29121,
    "training_samples": int(
        len(y_train_selected)
    ),
    "samples_per_class": int(
        np.sum(y_train_selected == 0)
    ),
    "completed_epochs": int(
        len(history_selected_df)
    ),
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(
        best_validation_loss
    ),
    "decision_threshold": float(
        LOCKED_THRESHOLD_50
    ),
    "threshold_selection": (
        "Maximum validation macro-F1, "
        "with balanced accuracy as tie-breaker"
    )
}

with open(
    RESULT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_config_50,
        file,
        indent=4
    )

print(
    "50% essential results saved to:",
    RESULT_DIR.resolve()
)

### 8.16 50% Test Results

The 50% real-data baseline was evaluated on the untouched test set using the validation-selected threshold of **0.4951**.

### Test Performance

| Metric | Result |
|---|---:|
| Accuracy | 0.9828 |
| Balanced accuracy | 0.9683 |
| Macro-F1 | 0.9651 |
| ROC-AUC | 0.9971 |
| Bird precision | 0.9329 |
| Bird recall | 0.9478 |
| Bird F1-score | 0.9403 |
| Drone precision | 0.9913 |
| Drone recall | 0.9887 |
| Drone F1-score | 0.9900 |

### Confusion Matrix

| True class | Predicted bird | Predicted drone | Total |
|---|---:|---:|---:|
| Bird | 945 | 52 | 997 |
| Drone | 68 | 5,931 | 5,999 |

The model correctly identifies:

- **945 of 997 birds**
- **5,931 of 5,999 drones**

It misclassifies:

- **52 birds as drones**
- **68 drones as birds**

The validation-selected threshold transfers successfully to the test set. The test balanced accuracy of 96.83% is close to the validation balanced accuracy of 97.32%.

### Real-Data Learning Curve: 10%, 25%, and 50%

| Metric | 10% | 25% | 50% |
|---|---:|---:|---:|
| Samples per class | 575 | 1,438 | 2,875 |
| Accuracy | 0.8942 | 0.9797 | 0.9828 |
| Balanced accuracy | 0.8497 | 0.9522 | 0.9683 |
| Macro-F1 | 0.8082 | 0.9579 | 0.9651 |
| ROC-AUC | 0.9353 | 0.9962 | 0.9971 |
| Bird precision | 0.5979 | 0.9421 | 0.9329 |
| Bird recall | 0.7874 | 0.9137 | 0.9478 |
| Bird F1 | 0.6797 | 0.9277 | 0.9403 |
| Drone recall | 0.9120 | 0.9907 | 0.9887 |

### Change from 25% to 50%

| Metric | Absolute change |
|---|---:|
| Accuracy | +0.0031 |
| Balanced accuracy | +0.0161 |
| Macro-F1 | +0.0072 |
| ROC-AUC | +0.0009 |
| Bird precision | -0.0092 |
| Bird recall | +0.0341 |
| Bird F1 | +0.0126 |
| Drone recall | -0.0020 |

Doubling the balanced training data from 1,438 to 2,875 samples per class produces a moderate improvement in balanced accuracy and bird recall.

Bird recall increases by 3.41 percentage points, reducing bird-to-drone errors from 86 to 52. However, bird precision decreases slightly because drone-to-bird errors increase from 56 to 68.

The improvement from 25% to 50% is much smaller than the improvement from 10% to 25%. This suggests a nonlinear learning curve with diminishing returns.

The 25% subset already provides strong class separation. Additional real data primarily improves minority-class sensitivity rather than producing a major increase in ROC-AUC.

In [ ]:
subtype_metrics_50 = (
    test_results_50_df
    .groupby(
        [
            "true_target_group",
            "original_label"
        ],
        observed=True
    )
    .agg(
        samples=("correct", "size"),
        correct_predictions=(
            "correct",
            "sum"
        ),
        recall=("correct", "mean"),
        mean_drone_probability=(
            "drone_probability",
            "mean"
        ),
        median_drone_probability=(
            "drone_probability",
            "median"
        )
    )
    .reset_index()
    .sort_values(
        [
            "true_target_group",
            "recall"
        ]
    )
)

display(
    subtype_metrics_50.style.format({
        "recall": "{:.2%}",
        "mean_drone_probability": "{:.4f}",
        "median_drone_probability": "{:.4f}"
    })
)

In [ ]:
test_results_50_df["range_bin"] = pd.cut(
    test_results_50_df["range_m"],
    bins=RANGE_BINS,
    labels=RANGE_LABELS,
    right=False,
    include_lowest=True
)

range_records_50 = []

for range_label in RANGE_LABELS:
    group = test_results_50_df[
        test_results_50_df["range_bin"]
        == range_label
    ]

    y_true = group[
        "true_binary_label"
    ].to_numpy()

    y_pred = group[
        "predicted_binary_label"
    ].to_numpy()

    probabilities = group[
        "drone_probability"
    ].to_numpy()

    range_records_50.append({
        "range_bin": range_label,
        "samples": len(group),

        "bird_samples": int(
            np.sum(y_true == 0)
        ),

        "drone_samples": int(
            np.sum(y_true == 1)
        ),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            y_pred[y_true == 0] == 0
        ),

        "drone_recall": np.mean(
            y_pred[y_true == 1] == 1
        ),

        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),

        "mean_drone_probability":
            probabilities.mean()
    })

range_metrics_50 = pd.DataFrame(
    range_records_50
)

display(range_metrics_50.round(4))

In [ ]:
subtype_metrics_50.to_csv(
    RESULT_DIR / "subtype_metrics.csv",
    index=False
)

range_metrics_50.to_csv(
    RESULT_DIR / "range_metrics.csv",
    index=False
)

print("Detailed 50% metrics saved.")

### 8.17 Subtype and Range Analysis

### Bird-Category Performance

| Bird category | Samples | Correct | Recall |
|---|---:|---:|---:|
| Seagull | 303 | 272 | 89.77% |
| Seagull and black-headed gull | 45 | 43 | 95.56% |
| Black-headed gull | 599 | 581 | 96.99% |
| Heron | 45 | 44 | 97.78% |
| Pigeon | 4 | 4 | 100.00% |
| Raven | 1 | 1 | 100.00% |

The two largest bird categories show strong performance:

- Seagull recall reaches **89.77%**.
- Black-headed gull recall reaches **96.99%**.

Together, seagulls and black-headed gulls represent 902 of the 997 bird samples. Therefore, their results provide the most reliable evidence of bird-class performance.

The model makes:

- 31 errors on seagulls
- 18 errors on black-headed gulls
- 2 errors on the mixed gull category
- 1 error on herons

The pigeon and raven results remain statistically unreliable because their supports are only four and one samples.

### Drone-Model Performance

| Drone model | Samples | Correct | Recall |
|---|---:|---:|---:|
| D3 | 1,000 | 970 | 97.00% |
| D1 | 1,000 | 976 | 97.60% |
| D5 | 1,000 | 991 | 99.10% |
| D4 | 999 | 996 | 99.70% |
| D2 | 1,000 | 998 | 99.80% |
| D6 | 1,000 | 1,000 | 100.00% |

Every drone model achieves at least **97% recall**. D3 remains the most difficult drone subtype, producing 30 of the 68 drone errors. D1 produces another 24 errors.

D2, D4, D5, and D6 all achieve recall above 99%.

### Comparison with the 25% Model

| Subtype | 25% recall | 50% recall | Change |
|---|---:|---:|---:|
| Seagull | 86.14% | 89.77% | +3.63 points |
| Black-headed gull | 94.82% | 96.99% | +2.17 points |
| Mixed gull category | 88.89% | 95.56% | +6.67 points |
| Heron | 82.22% | 97.78% | +15.56 points |
| D1 | 97.70% | 97.60% | -0.10 points |
| D3 | 97.10% | 97.00% | -0.10 points |

The additional real data primarily improves bird recognition. Drone-model recall was already close to saturation in the 25% experiment, so its small variations are not practically significant.

The large heron improvement corresponds to seven additional correct predictions. Because the heron test set contains only 45 samples, this change should be interpreted cautiously.

### Performance by Target Range

| Range interval | Balanced accuracy | Bird recall | Drone recall | ROC-AUC |
|---|---:|---:|---:|---:|
| <30 m | 0.9800 | 0.9619 | 0.9981 | 0.9989 |
| 30–50 m | 0.9742 | 0.9535 | 0.9949 | 0.9990 |
| 50–75 m | 0.9520 | 0.9291 | 0.9750 | 0.9924 |
| >75 m | 0.9603 | 0.9608 | 0.9598 | 0.9922 |

The 50% model achieves balanced accuracy above **95% in every range interval**.

The strongest performance occurs below 50 m, where balanced accuracy exceeds 97%. Performance remains robust beyond 75 m, with almost identical bird and drone recall:

- Bird recall: 96.08%
- Drone recall: 95.98%

This represents a strong correction of the range-dependent bias observed in the 10% experiment.

### Range Improvement from 25% to 50%

| Range | 25% balanced accuracy | 50% balanced accuracy | Change |
|---|---:|---:|---:|
| <30 m | 0.9651 | 0.9800 | +0.0149 |
| 30–50 m | 0.9611 | 0.9742 | +0.0131 |
| 50–75 m | 0.9330 | 0.9520 | +0.0190 |
| >75 m | 0.9346 | 0.9603 | +0.0257 |

Bird recall improves in every interval:

| Range | 25% bird recall | 50% bird recall | Change |
|---|---:|---:|---:|
| <30 m | 0.9333 | 0.9619 | +0.0286 |
| 30–50 m | 0.9267 | 0.9535 | +0.0268 |
| 50–75 m | 0.8830 | 0.9291 | +0.0461 |
| >75 m | 0.9020 | 0.9608 | +0.0588 |

The largest improvement occurs beyond 75 m, where bird recall increases by almost six percentage points.

Drone recall decreases slightly in the two longest intervals, but it remains above 95%. This reflects a small shift toward greater bird sensitivity rather than a substantial loss of drone recognition.

### 8.18 Diagnostic Conclusion

The 50% model provides strong and relatively uniform performance across target subtypes and range conditions.

Compared with the 25% model, the principal benefits are:

- Higher bird recall
- Better recognition of seagulls and black-headed gulls
- Better long-range balance
- Reduced dependence on target distance
- A decision threshold almost identical to 0.5

The improvements are smaller than those observed between 10% and 25%, confirming diminishing returns as the real training set grows.

The 50% model is now a strong real-only reference. Synthetic augmentation will be most valuable if it allows the 10% or 25% subsets to approach this performance without requiring the additional real observations.

## 9. 100% Real-Data Baseline

The final real-only baseline uses the complete balanced training subset:
5,749 bird and 5,749 drone segments (11,498 samples total). The architecture,
seed, data partitions, training procedure, and threshold-selection rule remain
unchanged.

### 9.1 Configuration

In [ ]:
SELECTED_SUBSET = "100_percent"
RUN_TRAINING = False

RESULT_DIR = (
    OUTPUT_DIR
    / f"{SELECTED_SUBSET}_seed_{RANDOM_SEED}"
)

CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / f"baseline_{SELECTED_SUBSET}_seed_{RANDOM_SEED}.keras"
)

print("Selected subset:", SELECTED_SUBSET)
print("Result directory:", RESULT_DIR.resolve())
print("Checkpoint:", CHECKPOINT_PATH.resolve())
print("Existing checkpoint:", CHECKPOINT_PATH.exists())

### 9.2 Overwrite Guard

In [ ]:
if RUN_TRAINING:
    if CHECKPOINT_PATH.exists():
        raise FileExistsError("The 100% checkpoint already exists.")
    if (RESULT_DIR / "test_metrics.csv").exists():
        raise FileExistsError("The 100% test results already exist.")
    RESULT_DIR.mkdir(parents=True, exist_ok=False)
else:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError("The saved 100% checkpoint was not found.")
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("The 100% experiment is ready.")

### 9.3 Load the Balanced Subset

In [ ]:
subset_indices = np.load(
    LIMITED_DATA_DIR
    / f"indices_{SELECTED_SUBSET}.npy"
)

X_train_selected = np.asarray(
    X_train_complete[subset_indices],
    dtype=np.float32
)[..., np.newaxis]

y_train_selected = np.asarray(
    y_train_complete[subset_indices],
    dtype=np.uint8
)

print(
    "Training tensor:",
    X_train_selected.shape
)

print(
    "Training labels:",
    y_train_selected.shape
)

print(
    "Class counts [bird, drone]:",
    np.bincount(y_train_selected)
)

print(
    "All values finite:",
    np.isfinite(X_train_selected).all()
)

print(
    "Value range:",
    float(X_train_selected.min()),
    "to",
    float(X_train_selected.max())
)

In [ ]:
assert X_train_selected.shape == (
    11498,
    5,
    150,
    1
)

assert y_train_selected.shape == (
    11498,
)

assert np.array_equal(
    np.bincount(y_train_selected),
    [5749, 5749]
)

assert np.isfinite(
    X_train_selected
).all()

assert X_train_selected.min() >= 0.0
assert X_train_selected.max() <= 1.0

print(
    "The 100% subset passed all checks."
)

### 9.4 Build the Training Dataset

In [ ]:
training_dataset_selected = (
    tf.data.Dataset
    .from_tensor_slices((
        X_train_selected,
        y_train_selected
    ))
    .shuffle(
        buffer_size=len(
            y_train_selected
        ),
        seed=RANDOM_SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(
    "Training batches:",
    len(training_dataset_selected)
)

### 9.5 Verify the Model Architecture

In [ ]:
tf.keras.backend.clear_session()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

model_selected = build_baseline_model()

total_parameters = (
    model_selected.count_params()
)

print(
    "Architecture parameters:",
    total_parameters
)

assert total_parameters == 29121

print(
    "Architecture verified: "
    "it matches all previous baselines."
)

### 9.6 Training Callbacks

In [ ]:
callbacks_selected = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        RESULT_DIR / "training_log.csv"
    )
]

### 9.7 Train or Load the Saved Model

In [ ]:
if RUN_TRAINING:
    history_selected = model_selected.fit(
        training_dataset_selected,
        validation_data=validation_dataset_fixed,
        epochs=50,
        callbacks=callbacks_selected,
        verbose=1
    )
    history_selected_df = pd.DataFrame(history_selected.history)
    history_selected_df.insert(
        0, "epoch", np.arange(1, len(history_selected_df) + 1)
    )
    history_selected_df.to_csv(
        RESULT_DIR / "training_history.csv", index=False
    )
else:
    history_path = RESULT_DIR / "training_history.csv"
    if not history_path.exists():
        raise FileNotFoundError("The saved 100% training history was not found.")
    model_selected = tf.keras.models.load_model(CHECKPOINT_PATH)
    history_selected_df = pd.read_csv(history_path)
    if "epoch" not in history_selected_df.columns:
        history_selected_df.insert(
            0, "epoch", np.arange(1, len(history_selected_df) + 1)
        )
    print("Saved 100% checkpoint loaded; no training was performed.")

assert model_selected.count_params() == 29121

### 9.8 Training Summary

In [ ]:
best_epoch = int(
    history_selected_df.loc[history_selected_df["val_loss"].idxmin(), "epoch"]
)
best_validation_loss = float(history_selected_df["val_loss"].min())

print("Completed epochs:", len(history_selected_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_validation_loss)
print("Checkpoint available:", CHECKPOINT_PATH.exists())

### 9.9 Training Results

The final real-only baseline was trained using the complete balanced subset:

- 5,749 bird segments
- 5,749 drone segments
- 11,498 total training samples
- 180 batches per epoch
- Maximum of 50 epochs
- Early-stopping patience of 8 epochs

Training stopped after epoch 18 and restored the model weights from epoch 10, which achieved the minimum validation loss.

### Best-Epoch Results

| Metric | Result |
|---|---:|
| Best epoch | 10 |
| Training loss | 0.0561 |
| Training accuracy | 0.9823 |
| Training ROC-AUC | 0.9974 |
| Validation loss | 0.0546 |
| Validation accuracy | 0.9825 |
| Validation ROC-AUC | 0.9975 |
| Validation PR-AUC | 0.9995 |

Training and validation performance are very close at the selected epoch. This indicates good segment-level generalization at the checkpoint chosen by validation loss.

The 100% model reaches its best validation result earlier than the 25% and 50% models. Validation performance fluctuates strongly after the best epoch, which confirms that early stopping remains necessary even when more training data is available.

### 9.10 Best Validation-Loss Comparison

| Training subset | Best epoch | Best validation loss |
|---|---:|---:|
| 10% | 6 | 0.3617 |
| 25% | 25 | 0.0619 |
| 50% | 29 | 0.0541 |
| 100% | 10 | 0.0546 |

The 100% model does not improve the minimum validation loss relative to the 50% model. The difference between 0.0541 and 0.0546 is very small and should not be interpreted as evidence that more data reduces performance.

Possible explanations include stochastic optimization, batch-normalization behaviour, the selected random seed, and saturation of the current model architecture.

Final conclusions must be based on the test metrics and, later, multiple random seeds.

### 9.11 Validation Probabilities

In [ ]:
validation_probabilities_100 = (
    model_selected
    .predict(
        validation_dataset_fixed,
        verbose=1
    )
    .reshape(-1)
)

print(
    "Minimum validation probability:",
    float(
        validation_probabilities_100.min()
    )
)

print(
    "Maximum validation probability:",
    float(
        validation_probabilities_100.max()
    )
)

print(
    "Mean validation probability:",
    float(
        validation_probabilities_100.mean()
    )
)

print(
    "Predictions below 0.5:",
    int(
        np.sum(
            validation_probabilities_100
            < 0.5
        )
    )
)

print(
    "Predictions at or above 0.5:",
    int(
        np.sum(
            validation_probabilities_100
            >= 0.5
        )
    )
)

### 9.12 Validation Threshold Search

In [ ]:
threshold_records_100 = []

for threshold in np.linspace(
    0.01,
    0.99,
    199
):
    predictions = (
        validation_probabilities_100
        >= threshold
    ).astype(np.uint8)

    threshold_records_100.append({
        "threshold": float(threshold),

        "accuracy": accuracy_score(
            y_validation_fixed,
            predictions
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_fixed,
                predictions
            ),

        "macro_f1": f1_score(
            y_validation_fixed,
            predictions,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            predictions[
                y_validation_fixed == 0
            ] == 0
        ),

        "drone_recall": np.mean(
            predictions[
                y_validation_fixed == 1
            ] == 1
        )
    })

threshold_search_100_df = pd.DataFrame(
    threshold_records_100
)

best_threshold_100_row = (
    threshold_search_100_df
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy"
        ],
        ascending=False
    )
    .iloc[0]
)

optimal_threshold_100 = float(
    best_threshold_100_row["threshold"]
)

print(
    "Selected validation threshold:",
    optimal_threshold_100
)

display(
    best_threshold_100_row
    .to_frame()
    .T
    .round(4)
)

### 9.13 Compare the Selected Threshold with 0.5

In [ ]:
threshold_comparison_100_records = []

for threshold_name, threshold in [
    ("default_0.5", 0.5),
    (
        "validation_optimised",
        optimal_threshold_100
    )
]:
    predictions = (
        validation_probabilities_100
        >= threshold
    ).astype(np.uint8)

    threshold_comparison_100_records.append({
        "threshold_name": threshold_name,
        "threshold": threshold,

        "accuracy": accuracy_score(
            y_validation_fixed,
            predictions
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_fixed,
                predictions
            ),

        "macro_f1": f1_score(
            y_validation_fixed,
            predictions,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            predictions[
                y_validation_fixed == 0
            ] == 0
        ),

        "drone_recall": np.mean(
            predictions[
                y_validation_fixed == 1
            ] == 1
        )
    })

threshold_comparison_100_df = (
    pd.DataFrame(
        threshold_comparison_100_records
    )
)

display(
    threshold_comparison_100_df.round(4)
)

### 9.14 Save Threshold Results

In [ ]:
threshold_search_100_df.to_csv(
    RESULT_DIR
    / "validation_threshold_search.csv",
    index=False
)

threshold_comparison_100_df.to_csv(
    RESULT_DIR
    / "validation_threshold_comparison.csv",
    index=False
)

print(
    "100% validation threshold results saved."
)

### 9.15 Validation Threshold Analysis

The 100% model produces validation probabilities across almost the complete probability interval:

- Minimum probability: approximately 0.000026
- Maximum probability: approximately 1.0000
- Mean probability: 0.8410
- Predictions below 0.5: 1,080
- Predictions at or above 0.5: 5,908

### Threshold Comparison

| Strategy | Threshold | Accuracy | Balanced accuracy | Macro-F1 | Bird recall | Drone recall |
|---|---:|---:|---:|---:|---:|---:|
| Default | 0.5000 | 0.9825 | 0.9831 | 0.9654 | 0.9838 | 0.9823 |
| Validation-optimized | 0.3218 | 0.9864 | 0.9790 | 0.9724 | 0.9687 | 0.9893 |

The optimized threshold increases:

- Accuracy from 98.25% to 98.64%
- Macro-F1 from 96.54% to 97.24%
- Drone recall from 98.23% to 98.93%

However, it decreases:

- Balanced accuracy from 98.31% to 97.90%
- Bird recall from 98.38% to 96.87%

This trade-off occurs because macro-F1 considers both precision and recall for both classes, whereas balanced accuracy considers only the average class recall.

The threshold of **0.3218** is retained because maximum validation macro-F1 was defined as the threshold-selection rule before evaluating the test set. The threshold is now locked and must not be modified using test results.

### 9.16 Predict the Untouched Test Set

In [ ]:
LOCKED_THRESHOLD_100 = float(
    optimal_threshold_100
)

print(
    "Locked 100% threshold:",
    LOCKED_THRESHOLD_100
)

test_probabilities_100 = (
    model_selected
    .predict(
        test_dataset_fixed,
        verbose=1
    )
    .reshape(-1)
)

test_predictions_100 = (
    test_probabilities_100
    >= LOCKED_THRESHOLD_100
).astype(np.uint8)

print(
    "Minimum test probability:",
    float(test_probabilities_100.min())
)

print(
    "Maximum test probability:",
    float(test_probabilities_100.max())
)

print(
    "Mean test probability:",
    float(test_probabilities_100.mean())
)

### 9.17 Calculate Test Metrics

In [ ]:
test_report_100 = classification_report(
    y_test_fixed,
    test_predictions_100,
    target_names=[
        "bird",
        "drone"
    ],
    output_dict=True,
    zero_division=0
)

test_metrics_100_df = pd.DataFrame([{
    "subset": SELECTED_SUBSET,
    "seed": RANDOM_SEED,
    "training_samples": len(
        y_train_selected
    ),
    "samples_per_class": int(
        np.sum(y_train_selected == 0)
    ),
    "best_epoch": best_epoch,
    "best_validation_loss":
        best_validation_loss,
    "threshold": LOCKED_THRESHOLD_100,

    "accuracy": accuracy_score(
        y_test_fixed,
        test_predictions_100
    ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test_fixed,
            test_predictions_100
        ),

    "macro_f1": f1_score(
        y_test_fixed,
        test_predictions_100,
        average="macro",
        zero_division=0
    ),

    "bird_precision":
        test_report_100["bird"]["precision"],

    "bird_recall":
        test_report_100["bird"]["recall"],

    "bird_f1":
        test_report_100["bird"]["f1-score"],

    "drone_precision":
        test_report_100["drone"]["precision"],

    "drone_recall":
        test_report_100["drone"]["recall"],

    "drone_f1":
        test_report_100["drone"]["f1-score"],

    "roc_auc": roc_auc_score(
        y_test_fixed,
        test_probabilities_100
    )
}])

display(
    test_metrics_100_df.round(4)
)

print(
    classification_report(
        y_test_fixed,
        test_predictions_100,
        target_names=[
            "bird",
            "drone"
        ],
        digits=4,
        zero_division=0
    )
)

### 9.18 Plot the Confusion Matrix

In [ ]:
confusion_matrix_100 = confusion_matrix(
    y_test_fixed,
    test_predictions_100
)

display(
    pd.DataFrame(
        confusion_matrix_100,
        index=[
            "True bird",
            "True drone"
        ],
        columns=[
            "Predicted bird",
            "Predicted drone"
        ]
    )
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    confusion_matrix_100,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Bird",
        "Drone"
    ],
    yticklabels=[
        "Bird",
        "Drone"
    ]
)

plt.title(
    "Test Confusion Matrix — 100% Real-Data Baseline"
)

plt.xlabel("Predicted target")
plt.ylabel("True target")
plt.tight_layout()
plt.show()

### 9.19 Save Essential Test Artifacts

In [ ]:
test_results_100_df = metadata_test.copy()

test_results_100_df[
    "true_binary_label"
] = y_test_fixed

test_results_100_df[
    "drone_probability"
] = test_probabilities_100

test_results_100_df[
    "predicted_binary_label"
] = test_predictions_100

test_results_100_df[
    "true_target_group"
] = np.where(
    y_test_fixed == 1,
    "drone",
    "bird"
)

test_results_100_df[
    "correct"
] = (
    y_test_fixed
    == test_predictions_100
)

test_metrics_100_df.to_csv(
    RESULT_DIR / "test_metrics.csv",
    index=False
)

test_results_100_df.to_csv(
    RESULT_DIR / "test_predictions.csv",
    index=False
)

experiment_config_100 = {
    "subset": SELECTED_SUBSET,
    "random_seed": RANDOM_SEED,
    "architecture_parameters": 29121,
    "training_samples": int(
        len(y_train_selected)
    ),
    "samples_per_class": int(
        np.sum(y_train_selected == 0)
    ),
    "completed_epochs": int(
        len(history_selected_df)
    ),
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(
        best_validation_loss
    ),
    "decision_threshold": float(
        LOCKED_THRESHOLD_100
    ),
    "threshold_selection": (
        "Maximum validation macro-F1, "
        "with balanced accuracy as tie-breaker"
    )
}

with open(
    RESULT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_config_100,
        file,
        indent=4
    )

print(
    "100% essential results saved to:",
    RESULT_DIR.resolve()
)

### 9.20 Untouched Test-Set Results

The final real-only model was evaluated on the untouched official test set using the validation-selected threshold of **0.3218**.

### Test Performance

| Metric | Result |
|---|---:|
| Accuracy | 0.9863 |
| Balanced accuracy | 0.9765 |
| Macro-F1 | 0.9722 |
| ROC-AUC | 0.9977 |
| Bird precision | 0.9421 |
| Bird recall | 0.9629 |
| Bird F1-score | 0.9524 |
| Drone precision | 0.9938 |
| Drone recall | 0.9902 |
| Drone F1-score | 0.9920 |

### Confusion Matrix

| True class | Predicted bird | Predicted drone | Total |
|---|---:|---:|---:|
| Bird | 960 | 37 | 997 |
| Drone | 59 | 5,940 | 5,999 |

The model correctly identifies:

- **960 of 997 bird samples**
- **5,940 of 5,999 drone samples**

It misclassifies:

- **37 birds as drones**
- **59 drones as birds**

Only 96 of the 6,996 test samples are misclassified. The validation-selected threshold transfers successfully to the test set, producing test balanced accuracy of 97.65% and macro-F1 of 97.22%.

### 9.21 Improvement from 50% to 100%

| Metric | 50% | 100% | Change |
|---|---:|---:|---:|
| Accuracy | 0.9828 | 0.9863 | +0.0035 |
| Balanced accuracy | 0.9683 | 0.9765 | +0.0082 |
| Macro-F1 | 0.9651 | 0.9722 | +0.0071 |
| ROC-AUC | 0.9971 | 0.9977 | +0.0006 |
| Bird precision | 0.9329 | 0.9421 | +0.0092 |
| Bird recall | 0.9478 | 0.9629 | +0.0151 |
| Bird F1 | 0.9403 | 0.9524 | +0.0121 |
| Drone recall | 0.9887 | 0.9902 | +0.0015 |

Doubling the balanced training set from 2,875 to 5,749 samples per class produces a further improvement, especially in bird recall.

Bird-to-drone errors decrease from 52 to 37, while drone-to-bird errors decrease from 68 to 59. Therefore, the 100% model improves both error directions.

However, the improvement remains much smaller than the improvement observed between the 10% and 25% subsets. The learning curve is approaching saturation for this architecture and official segment-level split.

### 9.22 Performance by Original Target Subtype

In [ ]:
subtype_metrics_100 = (
    test_results_100_df
    .groupby(
        [
            "true_target_group",
            "original_label"
        ],
        observed=True
    )
    .agg(
        samples=("correct", "size"),
        correct_predictions=("correct", "sum"),
        recall=("correct", "mean"),
        mean_drone_probability=(
            "drone_probability",
            "mean"
        ),
        median_drone_probability=(
            "drone_probability",
            "median"
        )
    )
    .reset_index()
    .sort_values(
        [
            "true_target_group",
            "recall"
        ]
    )
)

display(
    subtype_metrics_100.style.format({
        "recall": "{:.2%}",
        "mean_drone_probability": "{:.4f}",
        "median_drone_probability": "{:.4f}"
    })
)

### 9.23 Define Target-Range Intervals

In [ ]:
RANGE_BINS = [
    0,
    30,
    50,
    75,
    np.inf
]

RANGE_LABELS = [
    "<30 m",
    "30–50 m",
    "50–75 m",
    ">75 m"
]

print("Range intervals defined:")
print(RANGE_LABELS)

### 9.24 Performance by Target Range

In [ ]:
test_results_100_df["range_bin"] = pd.cut(
    test_results_100_df["range_m"],
    bins=RANGE_BINS,
    labels=RANGE_LABELS,
    right=False,
    include_lowest=True
)

range_records_100 = []

for range_label in RANGE_LABELS:
    group = test_results_100_df[
        test_results_100_df["range_bin"]
        == range_label
    ]

    y_true = group[
        "true_binary_label"
    ].to_numpy()

    y_pred = group[
        "predicted_binary_label"
    ].to_numpy()

    probabilities = group[
        "drone_probability"
    ].to_numpy()

    range_records_100.append({
        "range_bin": range_label,
        "samples": len(group),
        "bird_samples": int(
            np.sum(y_true == 0)
        ),
        "drone_samples": int(
            np.sum(y_true == 1)
        ),
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "bird_recall": np.mean(
            y_pred[y_true == 0] == 0
        ),
        "drone_recall": np.mean(
            y_pred[y_true == 1] == 1
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "mean_drone_probability":
            probabilities.mean()
    })

range_metrics_100 = pd.DataFrame(
    range_records_100
)

display(range_metrics_100.round(4))

### 9.25 Save Detailed Diagnostic Metrics

In [ ]:
subtype_metrics_100.to_csv(
    RESULT_DIR / "subtype_metrics.csv",
    index=False
)

range_metrics_100.to_csv(
    RESULT_DIR / "range_metrics.csv",
    index=False
)

print("Detailed 100% metrics saved.")

### 9.26 Subtype and Range Robustness

The 100% real-data model achieves strong performance across all four target-range intervals.

| Range | Balanced accuracy | Bird recall | Drone recall | ROC-AUC |
|---|---:|---:|---:|---:|
| <30 m | 0.9798 | 0.9619 | 0.9977 | 0.9991 |
| 30–50 m | 0.9784 | 0.9624 | 0.9943 | 0.9992 |
| 50–75 m | 0.9733 | 0.9681 | 0.9786 | 0.9947 |
| >75 m | 0.9557 | 0.9412 | 0.9702 | 0.9949 |

Balanced accuracy remains above **95% in every range interval**. The model is therefore substantially more robust to range variation than the original 10% baseline.

The strongest balanced performance occurs below 50 m. Beyond 75 m, balanced accuracy remains 95.57%, although this interval contains only 51 bird samples and therefore has greater uncertainty.

### Comparison with the 50% Model

| Range | 50% balanced accuracy | 100% balanced accuracy | Change |
|---|---:|---:|---:|
| <30 m | 0.9800 | 0.9798 | -0.0002 |
| 30–50 m | 0.9742 | 0.9784 | +0.0042 |
| 50–75 m | 0.9520 | 0.9733 | +0.0213 |
| >75 m | 0.9603 | 0.9557 | -0.0046 |

The largest improvement occurs at 50–75 m, where bird recall increases from 92.91% to 96.81%.

Performance below 30 m is effectively unchanged. Beyond 75 m, bird recall decreases slightly while drone recall increases. These small changes demonstrate that additional real data does not improve every subgroup monotonically under a single random seed.

### Range Conclusion

The 100% model no longer exhibits the severe short-range bird and long-range drone biases observed in the 10% experiment.

The remaining performance differences across range are relatively small. This supports the conclusion that broader training-data coverage improves range robustness.

Nevertheless, range intervals are correlated with target subtype and recording session. A future session-independent evaluation remains necessary.

## 10. Complete Real-Data Learning Curve

The four real-only experiments are now complete. Their saved test results are
combined below to quantify how classification performance changes with the
number of real training samples per class.

In [ ]:
SUBSET_ORDER = [
    "10_percent",
    "25_percent",
    "50_percent",
    "100_percent"
]

SAMPLES_PER_CLASS = {
    "10_percent": 575,
    "25_percent": 1438,
    "50_percent": 2875,
    "100_percent": 5749
}

TOTAL_TRAINING_SAMPLES = {
    subset: count * 2
    for subset, count
    in SAMPLES_PER_CLASS.items()
}

learning_curve_records = []

for subset_name in SUBSET_ORDER:
    metrics_path = (
        OUTPUT_DIR
        / f"{subset_name}_seed_{RANDOM_SEED}"
        / "test_metrics.csv"
    )

    if not metrics_path.exists():
        raise FileNotFoundError(
            f"Missing metrics: {metrics_path}"
        )

    metrics = pd.read_csv(
        metrics_path
    ).iloc[0].to_dict()

    metrics["subset"] = subset_name

    metrics["samples_per_class"] = (
        SAMPLES_PER_CLASS[subset_name]
    )

    metrics["training_samples"] = (
        TOTAL_TRAINING_SAMPLES[
            subset_name
        ]
    )

    learning_curve_records.append(
        metrics
    )

learning_curve_df = pd.DataFrame(
    learning_curve_records
)

learning_curve_df["subset"] = pd.Categorical(
    learning_curve_df["subset"],
    categories=SUBSET_ORDER,
    ordered=True
)

learning_curve_df = (
    learning_curve_df
    .sort_values("subset")
    .reset_index(drop=True)
)

display(
    learning_curve_df[
        [
            "subset",
            "samples_per_class",
            "training_samples",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "bird_precision",
            "bird_recall",
            "bird_f1",
            "drone_recall",
            "roc_auc"
        ]
    ].round(4)
)

In [ ]:
plot_metrics = {
    "balanced_accuracy":
        "Balanced accuracy",
    "macro_f1":
        "Macro-F1",
    "roc_auc":
        "ROC-AUC"
}

plt.figure(figsize=(10, 6))

for metric, display_name in (
    plot_metrics.items()
):
    plt.plot(
        learning_curve_df[
            "samples_per_class"
        ],
        learning_curve_df[metric],
        marker="o",
        linewidth=2,
        markersize=7,
        label=display_name
    )

plt.xlabel(
    "Number of real training samples per class"
)

plt.ylabel("Test score")

plt.title(
    "Real-Data Learning Curve\n"
    "Drone–Bird Micro-Doppler Classification"
)

plt.ylim(0.78, 1.01)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    learning_curve_df[
        "samples_per_class"
    ],
    learning_curve_df[
        "bird_recall"
    ],
    marker="o",
    linewidth=2,
    label="Bird recall"
)

plt.plot(
    learning_curve_df[
        "samples_per_class"
    ],
    learning_curve_df[
        "drone_recall"
    ],
    marker="o",
    linewidth=2,
    label="Drone recall"
)

plt.xlabel(
    "Number of real training samples per class"
)

plt.ylabel("Test recall")

plt.title(
    "Class-Specific Recall versus "
    "Real Training-Set Size"
)

plt.ylim(0.75, 1.01)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
improvement_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

improvement_df = (
    learning_curve_df[
        ["subset"] + improvement_columns
    ]
    .copy()
)

for metric in improvement_columns:
    improvement_df[
        f"{metric}_change"
    ] = (
        improvement_df[metric].diff()
    )

display(
    improvement_df[
        [
            "subset",
            "balanced_accuracy_change",
            "macro_f1_change",
            "bird_recall_change",
            "drone_recall_change",
            "roc_auc_change"
        ]
    ].round(4)
)

In [ ]:
learning_curve_df.to_csv(
    OUTPUT_DIR
    / "real_data_learning_curve.csv",
    index=False
)

improvement_df.to_csv(
    OUTPUT_DIR
    / "real_data_incremental_improvements.csv",
    index=False
)

print(
    "Complete real-data learning curve saved."
)

## 11. Conclusion — Real-Data Learning Curve

This experiment established the real-data reference learning curve for binary
drone-versus-bird micro-Doppler classification. The same CNN architecture,
official validation and test sets, training procedure, and random seed were used
for all four nested balanced training subsets. Only the amount of real training
data changed.

The strongest performance improvement occurred between the 10% and 25% subsets.
Increasing the training set from 575 to 1,438 samples per class improved test
balanced accuracy from 0.8497 to 0.9522 and macro-F1 from 0.8082 to 0.9579.
Bird recall increased from 0.7874 to 0.9137, while drone recall increased from
0.9120 to 0.9907.

Increasing the real-data set beyond 25% produced smaller but consistent gains.
At 50%, balanced accuracy reached 0.9683 and macro-F1 reached 0.9651. The complete
100% balanced real-data baseline achieved the best overall results, with an
accuracy of 0.9863, balanced accuracy of 0.9765, macro-F1 of 0.9722, bird recall
of 0.9629, drone recall of 0.9902, and ROC-AUC of 0.9977.

The learning curve therefore exhibits diminishing returns. Most of the available
performance gain was obtained when moving from 10% to 25% of the balanced real
training data. Additional data primarily improved bird recognition and reduced
the remaining class imbalance in recall, while drone recall and ROC-AUC were
already close to saturation.

These experiments provide the reference baselines required for evaluating
synthetic-data augmentation. The principal augmentation experiment will start
from the 10% real-data subset, where limited-data performance is weakest. Synthetic
samples will be added only to the training set, while the official real validation
and test sets will remain unchanged. The central question is whether augmented
10% training can approach or exceed the performance of the 25% real-only
baseline without introducing synthetic-to-real generalization errors.